# Cortical thickness

Combine Deep Atropos segmentation with Kelly–Kapowski (DiReCT) thickness estimation.

**Resources:** downloads an example T1, templates and pretrained weights on first use.
These 3-D workflows can take several minutes or longer and substantial memory on CPU.
Assets are cached under `~/.antstorch`. Inspect outputs on your own data.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

## Load the sample MRI or choose a local image

In [ ]:
from pathlib import Path

# Set an absolute path to use your own scalar 3-D T1 MRI instead.
t1_path = None
if t1_path is None:
    t1_path = antstorch.get_antstorch_data("mprage_hippmapp3r")
t1 = ants.image_read(str(t1_path)).clone("float")
assert t1.dimension == 3 and not t1.has_components
assert np.isfinite(t1.numpy()).all()
print("Input:", t1_path, "shape:", t1.shape, "spacing:", t1.spacing)

## Run the complete workflow
`backend="antstorch"` selects ANTsTorch for N4, registration and DiReCT.
This runs segmentation again independently of the Deep Atropos notebook.
Thickness is expressed in the physical units of the input spacing, typically mm.

In [ ]:
result = antstorch.cortical_thickness(
    t1, backend="antstorch", device=device, verbose=True,
)
thickness = result["thickness_image"]
segmentation = result["segmentation_image"]
assert thickness.shape == t1.shape
values = thickness.numpy()
assert np.isfinite(values).all()
assert (values >= 0).all()
assert (values > 0).any()
np.testing.assert_allclose(thickness.spacing, t1.spacing)
np.testing.assert_allclose(thickness.origin, t1.origin)
np.testing.assert_allclose(thickness.direction, t1.direction)
print("Median positive thickness:", float(np.median(values[values > 0])))

## Inspect the thickness map
Check both the tissue segmentation and the thickness overlay. A finite, positive
result alone does not establish anatomical accuracy. The median above summarizes
positive voxels only and is not an atlas-based cortical summary.

In [ ]:
ants.plot(t1, overlay=segmentation, axis=2, nslices=6,
          overlay_cmap="jet", overlay_alpha=0.5)
ants.plot(t1, overlay=thickness, axis=2, nslices=6,
          overlay_cmap="Reds", overlay_alpha=0.6)
output_dir = Path("example_outputs/cortical_thickness")
output_dir.mkdir(parents=True, exist_ok=True)
for name, image in result.items():
    ants.image_write(image, str(output_dir / f"{name}.nii.gz"))
print("Outputs (matching files are overwritten):", output_dir.resolve())